# Ginzburg-Landau 2D — FEniCS Benchmark

This notebook runs the FEniCS (DOLFINx) Ginzburg-Landau solvers and displays results.

**Prerequisites**: Run inside the devcontainer (or any environment where DOLFINx >= 0.10 is available).

## Problem

$$\min_u J(u) = \int_\Omega \frac{\varepsilon}{2} |\nabla u|^2 + \frac{1}{4}(u^2 - 1)^2 \, dx, \quad u|_{\partial\Omega} = 0$$

with $\varepsilon = 0.01$ on $[-1,1]^2$. This is a **non-convex** energy (indefinite Hessian), so not all solvers/settings are reliable.

## Solvers

- `GinzburgLandau2D_fenics/solve_GL_custom_jaxversion.py` — **Custom Newton (recommended)**: golden-section line search on $[-0.5, 2]$, GMRES + HYPRE AMG. Reliable for all mesh sizes and MPI decompositions.
- `GinzburgLandau2D_fenics/solve_GL_snes_newton.py` — **SNES Newton** (`newtontr`): trust-region Newton with FGMRES + ASM/ILU. Only reliable SNES configuration found (see `results_GinzburgLandau2D.md`).

> **Note**: GMRES is required (not CG) because the Ginzburg-Landau Hessian is indefinite.

Meshes are generated on the fly from the parameter `N = 2^(level+1)` subdivisions on $[-1,1]^2$.

In [ ]:
import subprocess, json, os, sys
import pandas as pd
from IPython.display import Markdown, display

# Must be run from the repo root so solver imports resolve correctly
print(f"Working directory: {os.getcwd()}")

## Helper: run a solver and collect JSON results

In [ ]:
def run_solver(script, nprocs=1, levels=None, extra_args=None):
    """Run a GL solver script (optionally with mpirun) and return parsed JSON results.
    
    Args:
        script:     Path to the solver script relative to repo root.
        nprocs:     Number of MPI processes (1 = serial).
        levels:     List of mesh levels to run. Default: [5, 6, 7, 8, 9].
        extra_args: Additional command-line arguments (e.g. ['--quiet']).
    """
    if levels is None:
        levels = [5, 6, 7, 8, 9]

    tmp_json = "/tmp/_gl_benchmark_result.json"

    # Build the command: optionally prefix with mpirun for parallel runs
    cmd = []
    if nprocs > 1:
        cmd = ["mpirun", "-n", str(nprocs)]
    cmd += ["python3", script, "--levels"] + [str(l) for l in levels]
    cmd += ["--json", tmp_json]
    if extra_args:
        cmd += extra_args

    print(f"Running: {' '.join(cmd)}")
    result = subprocess.run(cmd, capture_output=True, text=True)
    print(result.stdout)
    if result.returncode != 0:
        print(f"STDERR:\n{result.stderr}", file=sys.stderr)
        return None

    with open(tmp_json) as f:
        data = json.load(f)
    os.remove(tmp_json)
    return data

## 1. Custom Newton — Serial (recommended)

Re-implementation of the JAX minimiser on top of PETSc. Uses golden-section line search on $[-0.5, 2]$ with tol=1e-3, GMRES + HYPRE AMG with rtol=1e-3. Converges in **6 iterations** at all mesh levels.

In [ ]:
custom_serial = run_solver(
    "GinzburgLandau2D_fenics/solve_GL_custom_jaxversion.py",
    nprocs=1,
    levels=[5],           # Use level 5 for a quick demo; change to [5,6,7,8,9] for full benchmark
    extra_args=["--quiet"]  # Suppress per-iteration output
)
if custom_serial:
    df = pd.DataFrame(custom_serial["results"])
    df = df.rename(columns={"mesh_level": "lvl", "total_dofs": "dofs"})
    display(df[["lvl", "dofs", "time", "iters", "energy"]].to_markdown(index=False))

## 2. Custom Newton — Parallel (4 processes)

Same algorithm running under MPI. The golden-section line search is performed collectively across all processes.

In [ ]:
custom_4proc = run_solver(
    "GinzburgLandau2D_fenics/solve_GL_custom_jaxversion.py",
    nprocs=4,
    levels=[5],
    extra_args=["--quiet"]
)
if custom_4proc:
    df = pd.DataFrame(custom_4proc["results"])
    df = df.rename(columns={"mesh_level": "lvl", "total_dofs": "dofs"})
    display(df[["lvl", "dofs", "time", "iters", "energy"]].to_markdown(index=False))

## 3. SNES Newton — Serial

PETSc built-in trust-region Newton (`newtontr`) solver. Trust-region is required for non-convex Ginzburg-Landau because the Hessian is indefinite. Uses FGMRES + ASM/ILU(1) preconditioning with loose inner tolerance (ksp_rtol=1e-1).

> **Warning**: SNES with basic line search is unreliable for this non-convex problem — it may converge to wrong local minima or diverge, especially in parallel. `newtontr` is the most robust SNES variant found (see `results_GinzburgLandau2D.md`).

In [ ]:
snes_serial = run_solver(
    "GinzburgLandau2D_fenics/solve_GL_snes_newton.py",
    nprocs=1,
    levels=[5]
)
if snes_serial:
    df = pd.DataFrame(snes_serial["results"])
    df = df.rename(columns={"mesh_level": "lvl", "total_dofs": "dofs"})
    display(df[["lvl", "dofs", "time", "iters", "energy"]].to_markdown(index=False))

## 4. SNES Newton — Parallel (4 processes)

In [ ]:
snes_4proc = run_solver(
    "GinzburgLandau2D_fenics/solve_GL_snes_newton.py",
    nprocs=4,
    levels=[5]
)
if snes_4proc:
    df = pd.DataFrame(snes_4proc["results"])
    df = df.rename(columns={"mesh_level": "lvl", "total_dofs": "dofs"})
    display(df[["lvl", "dofs", "time", "iters", "energy"]].to_markdown(index=False))

## 5. Comparison Table

Side-by-side: Custom Newton vs SNES, serial vs 4-proc.
Both solvers should converge to the same energy value at each mesh level.

In [ ]:
def results_to_df(data, label):
    """Convert solver output dict to a DataFrame with a solver label column."""
    if data is None:
        return pd.DataFrame()
    df = pd.DataFrame(data["results"])
    df = df.rename(columns={"mesh_level": "lvl", "total_dofs": "dofs"})
    df["solver"] = label
    return df

frames = [
    results_to_df(custom_serial, "Custom serial"),
    results_to_df(custom_4proc,  "Custom 4-proc"),
    results_to_df(snes_serial,   "SNES serial"),
    results_to_df(snes_4proc,    "SNES 4-proc"),
]
frames = [f for f in frames if not f.empty]

if frames:
    all_df = pd.concat(frames, ignore_index=True)
    pivot = all_df.pivot_table(
        index=["lvl", "dofs"],
        columns="solver",
        values=["time", "iters"]
    )
    pivot.columns = [f"{c[1]} {c[0]}" for c in pivot.columns]
    display(Markdown(pivot.to_markdown()))

## 6. Full Benchmark (all levels)

To reproduce the full results from `results_GinzburgLandau2D.md`, run the automated experiment script:

```bash
python3 results_GL/run_experiments.py --nprocs 1 4 8 16 --levels 5 6 7 8 9 --repeat 3 --tag my_machine
```

Then generate Markdown/LaTeX tables:

```bash
python3 results_GL/generate_latex_tables.py results_GL/<experiment_id>/ --markdown
```

Or simply change `levels=[5]` to `levels=[5,6,7,8,9]` in the cells above.